# Bab 2 — Fundamental concepts: how do machines learn?

**Mahasiswa:** Tarannia Raisha Putri · 101032300231 · TK4702 · Deep Learning

**Referensi utama:** Andrew W. Trask, *Grokking Deep Learning*, Manning (2019), Bab 2, halaman cetak 9–19 (PDF 30–40, indeks PDF mulai 1). Subbab spesifik dicantumkan pada setiap tahap.

**Tujuan pembelajaran:** Membedakan empat kombinasi kategori pembelajaran serta menghubungkan parameter, target, prediksi, dan loss.

## Ringkasan

Bab ini memetakan deep learning di dalam machine learning. Dua sumbu berbeda digunakan: ketersediaan target (supervised/unsupervised) dan cara model merepresentasikan pola (parametric/nonparametric). Kerangka predict–compare–learn menjadi pola dasar seluruh bab implementasi.

## Konsep, intuisi, dan notasi

Dalam supervised learning tersedia pasangan $(x_i,y_i)$. Model parametrik memilih $\theta$ untuk mengurangi $L(\theta)=\frac1N\sum_i \ell(f_\theta(x_i),y_i)$. Rumus formal ini adalah **tambahan penjelasan**, bukan listing bab 2. Untuk data tabular $X\in\mathbb R^{N\times d}$, target regresi skalar $y\in\mathbb R^N$; dimensi parameter ditentukan model.

Unsupervised learning tidak menyediakan label target eksternal yang sama seperti klasifikasi, tetapi tetap memerlukan kriteria: misalnya kedekatan dalam kelompok atau kualitas rekonstruksi. Tanpa label bukan berarti tanpa tujuan.

Parametrik berarti bentuk model memiliki jumlah parameter yang ditentukan arsitektur. Nonparametrik tidak berarti tidak mempunyai parameter sama sekali; kapasitas representasinya dapat meningkat bersama data. Analogi “tombol” pada buku membantu membayangkan perubahan bobot tetapi tidak mencakup semua algoritme ML.

## Persiapan dan cara membaca

Jalankan **Restart Kernel → Run All**. Semua penjelasan merupakan parafrasa orisinal; angka hasil berada pada output sel yang benar-benar dijalankan. Label **reproduksi** berarti algoritme/contoh buku ditulis ulang; **koreksi/penyesuaian** menunjukkan perubahan; **tambahan** menunjukkan eksperimen atau interpretasi praktikum. `GDL_MODE=quick` mengurangi latihan berat; `full` memilih anggaran buku, dengan perubahan yang tetap dicatat. Keberhasilan eksekusi tidak membuktikan kesamaan akurasi dengan buku.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Jalankan dari root repository atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = os.environ.get('GDL_MODE', 'quick')
assert MODE in {'quick', 'full'}
FULL = MODE == 'full'
np.random.seed(1)
np.set_printoptions(precision=6, suppress=True)
METRICS = {}
def record(**values):
    METRICS.update(values)
def finite(*arrays):
    assert all(np.isfinite(np.asarray(a)).all() for a in arrays), 'NaN/inf terdeteksi'
def softmax(z):
    z = np.asarray(z)
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z)))
print('Mode:', MODE, '| NumPy:', np.__version__)

Mode: quick | NumPy: 2.5.4


<a id="b02-jenis"></a>

## Deep learning, machine learning, dan jenis supervisi

**Sumber:** cetak 10–13; PDF 31–34.

**Reproduksi konsep.** Machine learning mempelajari keteraturan melalui contoh. Deep learning adalah salah satu kelompok metode di dalamnya, terutama representasi berlapis. Supervised learning menghubungkan dataset masukan dengan keluaran yang diketahui; unsupervised learning menyingkap struktur tanpa pasangan jawaban tersebut.

Pada analogi harga saham dalam buku, nilai hari sebelumnya merupakan masukan dan nilai hari berikutnya target. Ini ilustrasi struktur tugas, bukan bukti bahwa prediksi akan akurat. Pada clustering, nomor cluster tidak otomatis memiliki arti semantik: arti kelompok harus ditelaah setelah pola ditemukan.

<a id="b02-parameter"></a>

## Parametric/nonparametric dan predict–compare–learn

**Sumber:** cetak 14–19; PDF 35–40.

**Reproduksi konsep; tabel contoh adalah tambahan klarifikasi.** Dua sumbu berikut tidak saling menggantikan.

| Supervisi | Parametrik | Nonparametrik |
|---|---|---|
| Supervised | Regresi linear dengan jumlah koefisien tetap | k-nearest neighbors yang menyimpan contoh |
| Unsupervised | k-means dengan jumlah centroid tetap | Estimasi kepadatan berbasis kernel |

Pada supervised parametric learning, **predict** memakai nilai parameter saat ini, **compare** menghitung selisih terhadap target, dan **learn** mengubah parameter. Membandingkan tanpa memperbarui bobot adalah evaluasi. Pada unsupervised parametric learning, parameter juga diubah tetapi sinyal pembanding berasal dari struktur data/tujuan yang dipilih. Model nonparametrik dapat memanfaatkan contoh dan perhitungan kemiripan tanpa menjalankan mekanisme pembaruan bobot yang sama.

Buku memakai penyederhanaan trial-and-error versus counting/probability. Itu alat orientasi, bukan definisi universal: model parametrik pun dapat menyatakan probabilitas dan nonparametrik pun dapat dioptimasi.

<a id="b02-demo"></a>

## Tambahan: mengamati satu siklus pembelajaran

**Sumber:** cetak 16–16; PDF 37–37.

Contoh sintetis ini dibuat untuk praktikum dan **bukan kode Bab 2**. Hanya satu bobot diubah; data dan target tetap. Bandingkan loss sebelum dan sesudah pembaruan untuk mengenali tiga langkah.

In [2]:
x, target, weight, alpha = 2., 3., 0.5, 0.1
prediction = x * weight
before = 0.5 * (prediction-target)**2
gradient = x * (prediction-target)
weight -= alpha * gradient
after = 0.5 * (x*weight-target)**2
print({'prediksi_awal':prediction, 'bobot_baru':weight, 'loss_awal':before, 'loss_akhir':after})
assert after < before
record(loss_before=before, loss_after=after)

{'prediksi_awal': 1.0, 'bobot_baru': 0.9, 'loss_awal': 2.0, 'loss_akhir': 0.72}


## Analisis output aktual — run tersimpan

Mode **quick**, eksekusi 2026-10-16T01:50:08.892591+07:00. Ringkasan ini dibuat dari output validator, bukan angka contoh dalam buku.

Pada contoh sintetis tambahan, half squared error turun dari 2.000 menjadi 0.720. Perubahan satu bobot mengurangi error satu titik; belum ada pengujian generalisasi.

## Kesimpulan dan pembahasan pertanyaan

Kategori supervisi menjelaskan sumber sinyal belajar, sedangkan parametrik menjelaskan bentuk representasi. Output tambahan menunjukkan pembaruan dapat menurunkan loss pada satu titik; itu belum mengukur generalisasi.

**Apakah semua deep learning supervised?** Tidak; representasi berlapis dapat belajar dengan berbagai sumber sinyal.

**Mengapa cluster 1 bukan otomatis “kucing”?** Angka hanyalah identitas kelompok, tanpa label makna dari luar.

**Apakah banyak parameter berarti nonparametrik?** Tidak. Ukuran besar tetap parametrik jika arsitekturnya menetapkan jumlah bobot.

In [3]:
finite(*[v for v in METRICS.values() if isinstance(v, (int, float))])
print('VALIDATION_METRICS=' + json.dumps(METRICS, sort_keys=True))

VALIDATION_METRICS={"loss_after": 0.72, "loss_before": 2.0}
